In [26]:
# downloading dataset for the customer churn predictions

import kagglehub
import shutil
import os

path = kagglehub.dataset_download("dhrubangtalukdar/telco-customer-churn-data")

destination = "../data"
os.makedirs(destination, exist_ok=True)

for file in os.listdir(path):
    shutil.copy(os.path.join(path, file), destination)

print("Copied to:", os.path.abspath(destination))

Copied to: /home/rohit/projects/customer_churn/data


* **Exploration Begins** 

In [27]:
# eda 

import pandas as pd 
import numpy as np


In [28]:
data = pd.read_csv('../data/synthetic_customer_churn_100k.csv')
df = data
df.shape

(100000, 9)

In [29]:
df.head(10)

,CustomerID,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
0,1,56,Female,68,147.58,Two year,Bank transfer,10052.03,No
1,2,69,Male,32,22.54,Month-to-month,Mailed check,686.78,No
2,3,46,Female,10,52.47,One year,Electronic check,537.88,No
3,4,32,Male,22,109.67,Month-to-month,Mailed check,2390.04,Yes
4,5,60,Female,54,130.98,Month-to-month,Credit card,7081.28,No
5,6,25,Female,29,111.55,Month-to-month,Credit card,3167.13,Yes
6,7,78,Female,20,63.23,One year,Electronic check,1307.22,No
7,8,38,Male,11,96.63,Month-to-month,Bank transfer,1093.32,Yes
8,9,56,Male,43,74.06,Month-to-month,Mailed check,3230.58,Yes
9,10,75,Other,62,109.39,One year,Electronic check,6799.64,No


In [30]:
df[df.Churn=='Yes']

,CustomerID,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
3,4,32,Male,22,109.67,Month-to-month,Mailed check,2390.04,Yes
5,6,25,Female,29,111.55,Month-to-month,Credit card,3167.13,Yes
7,8,38,Male,11,96.63,Month-to-month,Bank transfer,1093.32,Yes
8,9,56,Male,43,74.06,Month-to-month,Mailed check,3230.58,Yes
13,14,28,Female,17,121.38,Month-to-month,Mailed check,2061.90,Yes
...,...,...,...,...,...,...,...,...,...
99982,99983,35,Male,44,79.51,Month-to-month,Mailed check,3567.54,Yes
99983,99984,55,Female,50,12.78,Month-to-month,Mailed check,553.31,Yes
99989,99990,78,Female,50,94.66,Month-to-month,Electronic check,4765.99,Yes
99992,99993,61,Female,72,59.74,Month-to-month,Electronic check,4268.59,Yes


In [31]:
df.dtypes

CustomerID          int64
Age                 int64
Gender             object
Tenure              int64
MonthlyCharges    float64
Contract           object
PaymentMethod      object
TotalCharges      float64
Churn              object
dtype: object

In [32]:

df.isna().sum()

CustomerID        0
Age               0
Gender            0
Tenure            0
MonthlyCharges    0
Contract          0
PaymentMethod     0
TotalCharges      0
Churn             0
dtype: int64

In [33]:
df['Churn'].value_counts(normalize=False)

Churn
No     66856
Yes    33144
Name: count, dtype: int64

In [34]:
pd.crosstab(df['Contract'], df['Churn'], normalize='index')


Churn,No,Yes
Contract,,
Month-to-month,0.534426,0.465574
One year,0.832509,0.167491
Two year,0.831215,0.168785


In [35]:
pd.crosstab(df['MonthlyCharges'], df['Churn'], normalize='index')

Churn,No,Yes
MonthlyCharges,,
10.00,0.714286,0.285714
10.01,0.800000,0.200000
10.02,0.666667,0.333333
10.03,0.714286,0.285714
10.04,0.800000,0.200000
...,...,...
149.96,0.400000,0.600000
149.97,0.333333,0.666667
149.98,0.166667,0.833333


In [36]:
df.groupby('Churn')['MonthlyCharges'].mean()

Churn
No     72.845838
Yes    94.355298
Name: MonthlyCharges, dtype: float64

In [37]:
df.groupby('Churn')['Tenure'].mean()

Churn
No     39.321796
Yes    30.889784
Name: Tenure, dtype: float64

*the following are the most important features* 
* Age 
* Gender
* Tenure
* MonthlyCharges
* Contract
* TotalCharges
* Churn 



In [38]:
## splitting dataset into parts features (X) --> target (Y)


x = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes':1 , 'No':0})

print(x.head(10))
print(y.head(10))

   CustomerID  Age  Gender  Tenure  MonthlyCharges        Contract  \
0           1   56  Female      68          147.58        Two year   
1           2   69    Male      32           22.54  Month-to-month   
2           3   46  Female      10           52.47        One year   
3           4   32    Male      22          109.67  Month-to-month   
4           5   60  Female      54          130.98  Month-to-month   
5           6   25  Female      29          111.55  Month-to-month   
6           7   78  Female      20           63.23        One year   
7           8   38    Male      11           96.63  Month-to-month   
8           9   56    Male      43           74.06  Month-to-month   
9          10   75   Other      62          109.39        One year   

      PaymentMethod  TotalCharges  
0     Bank transfer      10052.03  
1      Mailed check        686.78  
2  Electronic check        537.88  
3      Mailed check       2390.04  
4       Credit card       7081.28  
5       Credi

In [39]:
# separating data into parts numerical and categorical features  

categorical_feat = x.select_dtypes(include=['object']).columns
numeric_feat = x.select_dtypes(include=['float','int64']).columns

print(categorical_feat)
print(numeric_feat)

Index(['Gender', 'Contract', 'PaymentMethod'], dtype='object')
Index(['CustomerID', 'Age', 'Tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')


In [40]:
''' setting up a pipeline to auto-transform the data and perform normalization/scaling and one-hot encoding
     the categorical features'''

from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numeric_feat),
        ('cat',OneHotEncoder(handle_unknown='ignore') , categorical_feat)
    ]
)

def build_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', preprocess),
        ('model', model)
    ])



In [41]:
## splitting the dataset into parts Test-train
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # VERY IMPORTANT (handles imbalance)
)

#### *Logistic_Regression*

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipe = build_pipeline(LogisticRegression(max_iter=1000, class_weight='balanced'))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')

print("F1 scores:", scores)
print("Mean F1:", scores.mean())

# Train final model
pipe.fit(X_train, y_train)

F1 scores: [0.60017867 0.60195429 0.59692605 0.60687981 0.59649687]
Mean F1: 0.6004871370905711


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [43]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = pipe.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[9110 4261]
 [1972 4657]]
              precision    recall  f1-score   support

           0       0.82      0.68      0.75     13371
           1       0.52      0.70      0.60      6629

    accuracy                           0.69     20000
   macro avg       0.67      0.69      0.67     20000
weighted avg       0.72      0.69      0.70     20000



log_reg or logistic_regression may have very bad acuraccy but is pretty good in recall so this is good for the current use case 

### ***Testing with Random Forest*** 

In [44]:
#  the model is lacking in accuracy so lets try some other algorithms to improve it
#  .. since logistic regression is clearly not suitable for this kind of problem

from sklearn.ensemble import RandomForestClassifier

randomForest_pipe = build_pipeline(
    RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
)

#  here n_estimators means the no of decison
#  trees we are gonna use and the random_state means a seed that does random shuffling and ensures reproducability
#  n_jobs is for no of cores to be used (CPU) , and verbose will show the progress 

rf_scores = cross_val_score(randomForest_pipe, X_train, y_train, cv=cv, scoring='f1')

print("RF F1 scores:", rf_scores)
print("RF Mean F1:", rf_scores.mean())
print("RF Std:", rf_scores.std())


randomForest_pipe.fit(X_train, y_train)

RF F1 scores: [0.6149288  0.62179732 0.61633919 0.62331407 0.61510377]
RF Mean F1: 0.6182966290147518
RF Std: 0.00354394551581541


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
